DATA 266 Homework 12
LLM Optimization with GPTQ and Knowledge Distillation
This notebook implements Homework 12 end to end.

Personal parameters

SID4 = 2046
SEED = 2046
SLICE = 46
HP_ID = 0
CLS_A = 6
CLS_B = 3

Part 1 uses the model shown in the instructor linked Hugging Face GPTQ documentation: Qwen/Qwen3-0.6B.

Part 2 uses the instructor provided fine tuned BERT teacher and distilbert-base-uncased as the student.

In [1]:
!pip install -q gptqmodel optimum accelerate datasets evaluate

In [1]:
import random
import numpy as np
import torch

SID4 = 2046
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6

CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("SID4:", SID4)
print("SEED:", SEED)
print("SLICE:", SLICE)
print("HP_ID:", HP_ID)
print("CLS_A:", CLS_A)
print("CLS_B:", CLS_B)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SID4: 2046
SEED: 2046
SLICE: 46
HP_ID: 0
CLS_A: 6
CLS_B: 3
CUDA available: True
GPU: Tesla T4


In [2]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0.dev20261123+cu132
Uninstalling torchaudio-2.11.0.dev20261123+cu132:
  Successfully uninstalled torchaudio-2.11.0.dev20261123+cu132


In [3]:
!pip install --pre torchaudio --index-url https://download.pytorch.org/whl/nightly/cu132

Looking in indexes: https://download.pytorch.org/whl/nightly/cu132
  Using cached torchaudio-2.11.0.dev20261123%2Bcu132-py3-none-manylinux_2_28_x86_64.whl.metadata (7.6 kB)
Using cached torchaudio-2.11.0.dev20261123%2Bcu132-py3-none-manylinux_2_28_x86_64.whl (1.8 MB)


In [4]:
import torch
import transformers
import gptqmodel

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("GPTQModel:", gptqmodel.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("All GPTQ dependencies imported successfully.")


INFO  LogBar: headless/CI mode; progress animations disabled.                   
WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.
INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True' for memory saving.


W1124 01:43:49.713000 5026 torch/utils/_pytree.py:646] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1124 01:43:49.790000 5026 torch/utils/_pytree.py:646] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.           
INFO    ┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐ │ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │ └─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘ GPT-QModel   : 7.12.2 Transformers : 5.24.0 Torch        : 2.15.0+cu132 Triton       : 3.9.0


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


PyTorch: 2.15.0+cu132
CUDA: 13.2
Transformers: 5.24.0
GPTQModel: 7.12.2
GPU: Tesla T4
All GPTQ dependencies imported successfully.


In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer
from huggingface_hub import model_info

BASE_MODEL_ID = "Qwen/Qwen3-0.6B"

model_details = model_info(BASE_MODEL_ID)
BASE_MODEL_REVISION = model_details.sha

print("Base model ID:", BASE_MODEL_ID)
print("Exact revision:", BASE_MODEL_REVISION)

wikitext = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1"
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    revision=BASE_MODEL_REVISION
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print()
print("WikiText2 loaded successfully")
print("Train samples:", len(wikitext["train"]))
print("Validation samples:", len(wikitext["validation"]))
print("Test samples:", len(wikitext["test"]))
print("Tokenizer:", tokenizer.__class__.__name__)

Base model ID: Qwen/Qwen3-0.6B
Exact revision: c1899de289a04d12100db370d81485cdf75e47ca


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

... (8 outputs left out)

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            


WikiText2 loaded successfully
Train samples: 36718
Validation samples: 3760
Test samples: 4358
Tokenizer: Qwen2Tokenizer


In [6]:
import gc
import torch
from transformers import AutoModelForCausalLM

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    revision=BASE_MODEL_REVISION,
    torch_dtype=torch.float16,
    device_map="cuda"
)

base_model.eval()

allocated_gb = torch.cuda.memory_allocated() / (1024 ** 3)
reserved_gb = torch.cuda.memory_reserved() / (1024 ** 3)

print("Base model loaded successfully")
print("Model:", BASE_MODEL_ID)
print("Revision:", BASE_MODEL_REVISION)
print("Device:", next(base_model.parameters()).device)
print("Dtype:", next(base_model.parameters()).dtype)
print("GPU memory allocated GB:", round(allocated_gb, 3))
print("GPU memory reserved GB:", round(reserved_gb, 3))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Base model loaded successfully
Model: Qwen/Qwen3-0.6B
Revision: c1899de289a04d12100db370d81485cdf75e47ca
Device: cuda:0
Dtype: torch.float16
GPU memory allocated GB: 1.11
GPU memory reserved GB: 1.564


In [7]:
import math
import torch

test_text = "\n\n".join(
    text for text in wikitext["test"]["text"]
    if text.strip()
)

encodings = tokenizer(
    test_text,
    return_tensors="pt"
)

print("WikiText2 test tokens:", encodings.input_ids.size(1))

@torch.inference_mode()
def calculate_perplexity(
    model,
    encodings,
    max_length=1024,
    stride=512
):
    model.eval()

    input_ids = encodings.input_ids
    seq_len = input_ids.size(1)

    nll_sum = 0.0
    token_count = 0
    prev_end = 0

    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)

        target_length = end - prev_end

        input_batch = input_ids[:, begin:end].to("cuda")
        target_batch = input_batch.clone()

        target_batch[:, :-target_length] = -100

        outputs = model(
            input_ids=input_batch,
            labels=target_batch
        )

        valid_tokens = target_length

        nll_sum += outputs.loss.item() * valid_tokens
        token_count += valid_tokens

        prev_end = end

        if end == seq_len:
            break

    perplexity = math.exp(nll_sum / token_count)

    return perplexity

print("Perplexity evaluator ready.")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (298938 > 131072). Running this sequence through the model will result in indexing errors


WikiText2 test tokens: 298938
Perplexity evaluator ready.


In [8]:
import time

torch.cuda.synchronize()
start_time = time.perf_counter()

base_perplexity = calculate_perplexity(
    base_model,
    encodings,
    max_length=1024,
    stride=512
)

torch.cuda.synchronize()
evaluation_time = time.perf_counter() - start_time

print("Base LLM evaluation complete")
print("Perplexity:", round(base_perplexity, 4))
print("Evaluation time seconds:", round(evaluation_time, 2))

Base LLM evaluation complete
Perplexity: 20.0273
Evaluation time seconds: 65.62


In [9]:
import time
import numpy as np
import torch

benchmark_prompts = [
    "The history of machine learning began",
    "A practical use of artificial intelligence is",
    "In a large distributed system",
    "The purpose of model compression is",
    "Language models can be deployed efficiently by"
]

@torch.inference_mode()
def benchmark_inference(
    model,
    tokenizer,
    prompts,
    warmup_runs=5,
    measured_runs=30,
    max_new_tokens=32
):
    model.eval()

    warmup_inputs = tokenizer(
        prompts[0],
        return_tensors="pt"
    ).to("cuda")

    for _ in range(warmup_runs):
        model.generate(
            **warmup_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    latencies = []

    for i in range(measured_runs):
        prompt = prompts[i % len(prompts)]

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to("cuda")

        torch.cuda.synchronize()
        start = time.perf_counter()

        model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

        torch.cuda.synchronize()

        elapsed_ms = (time.perf_counter() - start) * 1000
        latencies.append(elapsed_ms)

    p50 = np.percentile(latencies, 50)
    p95 = np.percentile(latencies, 95)

    peak_memory_gb = (
        torch.cuda.max_memory_allocated() / (1024 ** 3)
    )

    return p50, p95, peak_memory_gb, latencies


base_p50, base_p95, base_peak_memory, base_latencies = benchmark_inference(
    base_model,
    tokenizer,
    benchmark_prompts
)

print("Base LLM inference benchmark complete")
print("Measured runs:", len(base_latencies))
print("p50 latency ms:", round(base_p50, 2))
print("p95 latency ms:", round(base_p95, 2))
print("Peak GPU memory GB:", round(base_peak_memory, 3))

Base LLM inference benchmark complete
Measured runs: 30
p50 latency ms: 1323.85
p95 latency ms: 1732.72
Peak GPU memory GB: 1.123


In [10]:
import os
import shutil
from pathlib import Path

BASE_SAVE_DIR = Path("/content/base_model_fp16")

if BASE_SAVE_DIR.exists():
    shutil.rmtree(BASE_SAVE_DIR)

base_model.save_pretrained(BASE_SAVE_DIR)
tokenizer.save_pretrained(BASE_SAVE_DIR)

base_size_bytes = sum(
    file.stat().st_size
    for file in BASE_SAVE_DIR.rglob("*")
    if file.is_file()
)

base_size_mb = base_size_bytes / (1024 ** 2)

print("Base model saved successfully")
print("Saved directory:", BASE_SAVE_DIR)
print("Size on disk MB:", round(base_size_mb, 2))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Base model saved successfully
Saved directory: /content/base_model_fp16
Size on disk MB: 1147.81
